In [7]:
import yaml

with open("C:/Users/Lales/.dbt/profiles.yml", 'r') as stream:
    data_loaded = yaml.safe_load(stream)

DBCONFIG = data_loaded["classicmodels_modeling"]["outputs"]["dev"]

DBHOST = DBCONFIG["host"]
DBPORT = int(DBCONFIG["port"])
DBNAME = DBCONFIG["dbname"]
DBUSER = DBCONFIG["user"]
DBPASSWORD = DBCONFIG["pass"]

db_connection_url = f'postgresql+psycopg2://{DBUSER}:{DBPASSWORD}@{DBHOST}:{DBPORT}/{DBNAME}'

%load_ext sql
%sql {db_connection_url}


<a id="1-1-2"></a>
<span style="font-size:16px"><b>1.1.2.</b></span> Initiate the `classicmodels_modeling` project with the `init` command:

```bash
dbt init classicmodels_modeling
```

In [8]:
from sqlalchemy import create_engine
import pandas as pd

engine = create_engine(db_connection_url)

df = pd.read_sql("""
SELECT *
FROM information_schema.tables
WHERE table_schema = 'public'
""", engine)

df

,table_catalog,table_schema,table_name,table_type,self_referencing_column_name,reference_generation,user_defined_type_catalog,user_defined_type_schema,user_defined_type_name,is_insertable_into,is_typed,commit_action
0,Student,public,productlines,BASE TABLE,None,None,None,None,None,YES,NO,None
1,Student,public,products,BASE TABLE,None,None,None,None,None,YES,NO,None
2,Student,public,employees,BASE TABLE,None,None,None,None,None,YES,NO,None
3,Student,public,offices,BASE TABLE,None,None,None,None,None,YES,NO,None
4,Student,public,customers,BASE TABLE,None,None,None,None,None,YES,NO,None
5,Student,public,payments,BASE TABLE,None,None,None,None,None,YES,NO,None
6,Student,public,orders,BASE TABLE,None,None,None,None,None,YES,NO,None
7,Student,public,orderdetails,BASE TABLE,None,None,None,None,None,YES,NO,None


# Star Schem Modeling

### 1 - Creating the Facts Table

In [ ]:


engine = create_engine(db_connection_url)

df = pd.read_sql("""
SELECT 
    orders.orderNumber, orderdetails.orderLineNumber,
    customers.customerNumber AS customer_key, 
    employees.employeeNumber AS employee_key,
   
    productCode AS product_key, 
    orders.orderDate AS order_date,
    orders.requiredDate AS order_required_date, 
    orders.shippedDate AS order_shipped_date,
    orderdetails.quantityOrdered AS quantity_ordered, 
    orderdetails.priceEach AS product_price
FROM public.orders
JOIN public.orderdetails ON orders.orderNumber = orderdetails.orderNumber
JOIN public.customers ON orders.customerNumber = customers.customerNumber
JOIN public.employees ON customers.salesRepEmployeeNumber = employees.employeeNumber
""", engine)
df

,ordernumber,orderlinenumber,customer_key,employee_key,product_key,order_date,order_required_date,order_shipped_date,quantity_ordered,product_price
0,10100,3,363,1216,S18_1749,2003-01-06,2003-01-13,2003-01-10,30,136.00
1,10100,2,363,1216,S18_2248,2003-01-06,2003-01-13,2003-01-10,50,55.09
2,10100,4,363,1216,S18_4409,2003-01-06,2003-01-13,2003-01-10,22,75.46
3,10100,1,363,1216,S24_3969,2003-01-06,2003-01-13,2003-01-10,49,35.29
4,10101,4,128,1504,S18_2325,2003-01-09,2003-01-18,2003-01-11,25,108.06
...,...,...,...,...,...,...,...,...,...,...
2991,10425,9,119,1370,S24_2300,2005-05-31,2005-06-07,None,49,127.79
2992,10425,5,119,1370,S24_2840,2005-05-31,2005-06-07,None,31,31.82
2993,10425,11,119,1370,S32_1268,2005-05-31,2005-06-07,None,41,83.79
2994,10425,6,119,1370,S32_2509,2005-05-31,2005-06-07,None,11,50.32


### 2 - Creating the Customers Dimension Table

In [22]:
engine = create_engine(db_connection_url)

df = pd.read_sql(""" 
    select customerNumber as customer_key, 
    customerName as customer_name,   
    contactLastName as customer_last_name, 
    contactFirstName as customer_first_name, 
    phone as phone, 
    addressLine1 as address_line_1, 
    addressLine2 as address_line_2, 
    postalCode as postal_code, 
    city as city, 
    state as state, 
    country as country,
    creditLimit as credit_limit
FROM public.customers""", engine)

df


,customer_key,customer_name,customer_last_name,customer_first_name,phone,address_line_1,address_line_2,postal_code,city,state,country,credit_limit
0,103,Atelier graphique,Schmitt,Carine,40.32.2555,"54, rue Royale",NaN,44000,Nantes,NaN,France,21000.0
1,112,Signal Gift Stores,King,Jean,7025551838,8489 Strong St.,NaN,83030,Las Vegas,NV,USA,71800.0
2,114,"Australian Collectors, Co.",Ferguson,Peter,03 9520 4555,636 St Kilda Road,Level 3,3004,Melbourne,Victoria,Australia,117300.0
3,119,La Rochelle Gifts,Labrune,Janine,40.67.8555,"67, rue des Cinquante Otages",NaN,44000,Nantes,NaN,France,118200.0
4,121,Baane Mini Imports,Bergulfsen,Jonas,07-98 9555,Erling Skakkes gate 78,NaN,4110,Stavern,NaN,Norway,81700.0
...,...,...,...,...,...,...,...,...,...,...,...,...
117,486,Motor Mint Distributors Inc.,Salazar,Rosa,2155559857,11328 Douglas Av.,NaN,71270,Philadelphia,PA,USA,72600.0
118,487,Signal Collectibles Ltd.,Taylor,Sue,4155554312,2793 Furth Circle,NaN,94217,Brisbane,CA,USA,60300.0
119,489,"Double Decker Gift Stores, Ltd",Smith,Thomas,(171) 555-7555,120 Hanover Sq.,NaN,WA1 1DP,London,NaN,UK,43300.0
120,495,Diecast Collectables,Franco,Valarie,6175552555,6251 Ingle Ln.,NaN,51003,Boston,MA,USA,85100.0


### 3 - Creating the Employees Dimension Table

In [11]:
engine = create_engine(db_connection_url)

df = pd.read_sql(""" SELECT
    employeeNumber as employee_key,
    lastName as employee_last_name, 
    firstName as employee_first_name, 
    jobTitle as job_title, 
    email as email
FROM public.employees
""", engine)

df


,employee_key,employee_last_name,employee_first_name,job_title,email
0,1002,Murphy,Diane,President,dmurphy@classicmodelcars.com
1,1056,Patterson,Mary,VP Sales,mpatterso@classicmodelcars.com
2,1076,Firrelli,Jeff,VP Marketing,jfirrelli@classicmodelcars.com
3,1088,Patterson,William,Sales Manager (APAC),wpatterson@classicmodelcars.com
4,1102,Bondur,Gerard,Sale Manager (EMEA),gbondur@classicmodelcars.com
5,1143,Bow,Anthony,Sales Manager (NA),abow@classicmodelcars.com
6,1165,Jennings,Leslie,Sales Rep,ljennings@classicmodelcars.com
7,1166,Thompson,Leslie,Sales Rep,lthompson@classicmodelcars.com
8,1188,Firrelli,Julie,Sales Rep,jfirrelli@classicmodelcars.com
9,1216,Patterson,Steve,Sales Rep,spatterson@classicmodelcars.com


### 5 - Creating the Office Dimension Table

In [13]:
engine = create_engine(db_connection_url)

df = pd.read_sql(""" SELECT 
    officeCode as office_key, 
    postalCode as postal_code, 
    city as city, 
    state as state, 
    country as country, 
    territory as territory
FROM public.offices
""", engine)

df

,office_key,postal_code,city,state,country,territory
0,1,94080,San Francisco,CA,USA,NA
1,2,02107,Boston,MA,USA,NA
2,3,10022,NYC,NY,USA,NA
3,4,75017,Paris,NaN,France,EMEA
4,5,102-8578,Tokyo,Chiyoda-Ku,Japan,Japan
5,6,NSW 2010,Sydney,NaN,Australia,APAC
6,7,EC2N 1HN,London,NaN,UK,EMEA


### 6 - Creating the Product Dimension Table

In [14]:
engine = create_engine(db_connection_url)

df = pd.read_sql(""" SELECT 
    productCode as product_key, 
    productName as product_name, 
    products.productLine as product_line, 
    productScale as product_scale, 
    productVendor as product_vendor,
    productDescription as product_description, 
    textDescription as product_line_description
FROM public.products
JOIN public.productlines ON products.productLine=productlines.productLine
""", engine)

df

,product_key,product_name,product_line,product_scale,product_vendor,product_description,product_line_description
0,S10_1678,1969 Harley Davidson Ultimate Chopper,Motorcycles,1:10,Min Lin Diecast,"This replica features working kickstand, front...",Our motorcycles are state of the art replicas ...
1,S10_1949,1952 Alpine Renault 1300,Classic Cars,1:10,Classic Metal Creations,Turnable front wheels; steering function; deta...,Attention car enthusiasts: Make your wildest c...
2,S10_2016,1996 Moto Guzzi 1100i,Motorcycles,1:10,Highway 66 Mini Classics,"Official Moto Guzzi logos and insignias, saddl...",Our motorcycles are state of the art replicas ...
3,S10_4698,2003 Harley-Davidson Eagle Drag Bike,Motorcycles,1:10,Red Start Diecast,"Model features, official Harley Davidson logos...",Our motorcycles are state of the art replicas ...
4,S10_4757,1972 Alfa Romeo GTA,Classic Cars,1:10,Motor City Art Classics,Features include: Turnable front wheels; steer...,Attention car enthusiasts: Make your wildest c...
...,...,...,...,...,...,...,...
105,S700_3505,The Titanic,Ships,1:700,Carousel DieCast Legends,"Completed model measures 19 1/2 inches long, 9...",The perfect holiday or anniversary gift for ex...
106,S700_3962,The Queen Mary,Ships,1:700,Welly Diecast Productions,Exact replica. Wood and Metal. Many extras inc...,The perfect holiday or anniversary gift for ex...
107,S700_4002,American Airlines: MD-11S,Planes,1:700,Second Gear Diecast,Polished finish. Exact replia with official lo...,"Unique, diecast airplane and helicopter replic..."
108,S72_1253,Boeing X-32A JSF,Planes,1:72,Motor City Art Classics,"10\"" Wingspan with retractable landing gears.C...","Unique, diecast airplane and helicopter replic..."


###   Running the Star Schema Model

```bash
dbt run -s star_schema
```

### Checking

In [17]:
engine = create_engine(db_connection_url)

df = pd.read_sql("""
SELECT * FROM information_schema.tables 
WHERE table_schema = 'public_star_schema'

""", engine)

df

,table_catalog,table_schema,table_name,table_type,self_referencing_column_name,reference_generation,user_defined_type_catalog,user_defined_type_schema,user_defined_type_name,is_insertable_into,is_typed,commit_action
0,Student,public_star_schema,dates,BASE TABLE,None,None,None,None,None,YES,NO,None
1,Student,public_star_schema,dim_offices,BASE TABLE,None,None,None,None,None,YES,NO,None
2,Student,public_star_schema,dim_dates,BASE TABLE,None,None,None,None,None,YES,NO,None
3,Student,public_star_schema,dim_customers,BASE TABLE,None,None,None,None,None,YES,NO,None
4,Student,public_star_schema,dim_products,BASE TABLE,None,None,None,None,None,YES,NO,None
5,Student,public_star_schema,dim_employees,BASE TABLE,None,None,None,None,None,YES,NO,None
6,Student,public_star_schema,fact_orders,BASE TABLE,None,None,None,None,None,YES,NO,None


In [19]:
df = pd.read_sql("""SELECT count(*) FROM public_star_schema.fact_orders;""", engine)

df

,count
0,2996


## One Big Table (OBT)

As the name suggests, it means a large table containing all the relevant data needed for analysis. It is similar to a fact table, but instead of using dimensional tables and foreign keys, it contains the required dimensional values for each row within. This approach ensures the data warehouse doesn't have to perform any joins to query the relevant data each time you need it. Here is an example of an OBT table focused on the orders of `classicmodels`:

<span style="font-size:16px"><b>4.1.</b></span> Create the file `orders_obt.sql` in the `./classicmodels_modeling/models/obt/` folder. 

<span style="font-size:16px"><b>4.4.</b></span> Make sure you are in the `~/project/classicmodels_modeling` folder in the terminal. Run the following command:

```shell
dbt run --select "obt"
```

df = pd.read_sql("""SELECT * FROM information_schema.tables 
WHERE table_schema = 'classicmodels_obt'""", engine)

df

In [21]:
df = pd.read_sql("""SELECT * FROM information_schema.tables 
WHERE table_schema = 'public_obt'""", engine)

df

,table_catalog,table_schema,table_name,table_type,self_referencing_column_name,reference_generation,user_defined_type_catalog,user_defined_type_schema,user_defined_type_name,is_insertable_into,is_typed,commit_action
0,Student,public_obt,oreders_obt,BASE TABLE,None,None,None,None,None,YES,NO,None


In [24]:
df = pd.read_sql("""SELECT count(*) FROM public_obt.oreders_obt;""", engine)

df

,count
0,2996
